Bradley–Terry: Where the DPO Preference Loss Starts

Don't think of Bradley–Terry as some huge ML algorithm. It's basically a mathematical model for pairwise preferences.

We want to model:

Given two answers, what's the probability that a human prefers A over B?

1. Give every response a hidden score

Suppose:

Prompt: Explain gravity.

Two responses:

\(A\): clear, correct explanation
\(B\): vague explanation

We assume each response has some latent reward:

$$ r(x,A) $$ $$ r(x,B) $$

Maybe:

$$ r(x,A)=3 $$ $$ r(x,B)=1 $$

The actual numbers don't matter much. The difference matters.

2. Preference should depend on the difference

If:

$$ r_A=r_B $$

we'd expect roughly:

$$ P(A\succ B)=0.5 $$

If:

$$ r_A>r_B $$

then:

$$ P(A\succ B)>0.5 $$

If:

$$ r_A\gg r_B $$

then:

$$ P(A\succ B)\rightarrow1 $$

So Bradley–Terry defines:

$$ \boxed{ P(A\succ B) = \sigma(r_A-r_B) } $$

where:

$$ \sigma(z)=\frac{1}{1+e^{-z}} $$
3. Tiny numerical example

Suppose:

$$ r_A=3 $$ $$ r_B=1 $$

Then:

$$ r_A-r_B=2 $$

Therefore:

$$ P(A\succ B) = \sigma(2) \approx0.881 $$

So the model says:

There's an 88.1% probability that A is preferred over B.

If instead:

$$ r_A=1,\quad r_B=3 $$

then:

$$ P(A\succ B)=\sigma(-2)\approx0.119 $$

Makes sense.

4. Why use the sigmoid?

Because we need a number between 0 and 1.

The reward difference can be:

$$ -\infty < r_A-r_B < +\infty $$

but probability must be:

$$ 0\le P\le1 $$

Sigmoid maps:

$$ (-\infty,+\infty) \rightarrow (0,1) $$

That's basically the reason.

5. Now bring in a dataset

Suppose humans give us:

$$ (x_i,y_{w,i},y_{l,i}) $$

Meaning:

For prompt \(x_i\), the human preferred \(y_w\) over \(y_l\).

Our reward model predicts:

$$ r_w=r(x_i,y_w) $$ $$ r_l=r(x_i,y_l) $$

Therefore:

$$ P(y_w\succ y_l|x_i) = \sigma(r_w-r_l) $$

We want this probability to be high.

6. Turn that into a loss

Maximum likelihood says:

Make the observed human preference as probable as possible.

For one example:

$$ P=\sigma(r_w-r_l) $$

Negative log-likelihood:

$$ \boxed{ L=-\log\sigma(r_w-r_l) } $$

That's the basic preference-learning loss.

Notice how simple it is:

$$ \boxed{ \text{reward chosen} - \text{reward rejected} \rightarrow \text{sigmoid} \rightarrow -\log } $$
7. What does the loss actually do?

Suppose:

$$ r_w-r_l=2 $$

Then:

$$ L=-\log(0.881)\approx0.127 $$

Pretty small.

The model already thinks the chosen response is better.

Now suppose:

$$ r_w-r_l=-2 $$

Then:

$$ P(w)=0.119 $$

and:

$$ L=-\log(0.119)\approx2.127 $$

Large loss.

So the model gets strongly pushed toward:

$$ r_w>r_l $$
8. The crucial limitation

Bradley–Terry only tells us about relative reward.

If:

$$ r_A=5,\quad r_B=3 $$

and:

$$ r_A=105,\quad r_B=103 $$

both have:

$$ r_A-r_B=2 $$

Therefore they produce the same preference probability.

So preference data doesn't tell us:

"A has an absolute quality of 5."

It tells us:

"A is better than B."

This becomes important when we later derive DPO.

9. Now connect this to DPO

At the moment Bradley–Terry says:

$$ P(y_w\succ y_l|x) = \sigma( r(x,y_w)-r(x,y_l) ) $$

But there's a problem:

Where do we get \(r(x,y)\)?

Classical RLHF:

preference pairs
      ↓
train reward model
      ↓
r(x,y)
      ↓
PPO

DPO's trick is essentially:

We can express that reward in terms of the policy itself.

First: yes, \(r\) is NOT a reward model in DPO

When we write:

$$ r(x,y) $$

right now, just think:

"some imaginary score that represents how good humans think this answer is."

We're using it to do the math. We're not training a reward network.

"Where did the probability come from?"

This is the important part.

Suppose a human gives us:

Question: Explain gravity.

Answer A → 👍 CHOSEN
Answer B → 👎 REJECTED

The human gave us one binary decision.

There is no probability in the dataset.

Correct.

But we want to build a mathematical model that predicts the probability of that decision.

That's where Bradley-Terry comes in.

Imagine the imaginary reward scores

We say:

A → reward = 3
B → reward = 1

Then we say:

If A's score is higher than B's, we're more likely to observe the human choosing A.

So:

$$ P(A\text{ chosen over }B) = \sigma(3-1) $$ $$ =\sigma(2) $$ $$ \approx0.88 $$

That probability is NOT from the human.

It's the model's predicted probability of the human preference.

That's the key distinction.

Think about normal binary classification

This is exactly the same idea.

Suppose training data says:

email → SPAM

The dataset only says:

$$ y=1 $$

But your neural network outputs:

$$ P(\text{spam})=0.93 $$

Where did 0.93 come from?

The neural network.

The label was only:

$$ 1 $$

Same thing here.

Human gives:

$$ y=1 $$

meaning:

$$ A\succ B $$

Bradley-Terry gives us a model that predicts:

$$ P(A\succ B)=0.88 $$
Now where does \(\pi\) come from?

THIS is the next important distinction.

$$ \pi(y|x) $$

is simply the LLM itself.

Remember what an LLM does:

Prompt
  ↓
Transformer
  ↓
probability of next token

For example:

"The sky is"

The model might output:

blue    0.72
green   0.03
dark    0.10
...

Those probabilities come directly from the model's softmax output.

For an entire response:

$$ \pi(y|x) $$

means:

"How probable does this LLM think this entire response is given this prompt?"

Example

Prompt:

Explain gravity.

Response:

"Gravity is the force that attracts objects with mass toward each other."

The LLM assigns probabilities to every token:

Gravity → 0.12
is      → 0.41
the     → 0.63
force   → 0.21
...

The probability of the whole response is the product of those token probabilities:

$$ \pi(y|x) = P(y_1|x) P(y_2|x,y_1) P(y_3|x,y_1,y_2) \cdots $$

Usually we work with log probabilities:

logπ(y∣x)=
t
∑
	​

logP(y
t
	​

∣previous tokens)

So now we have TWO completely different probabilities

This is where things get much clearer.

1. Human preference probability

Bradley-Terry says:

$$ P(A\succ B) $$

"Given their scores, how likely is A to be preferred?"

2. LLM probability
$$ \pi(y|x) $$

"How likely is the LLM to generate this response?"

These are not the same thing.

And DPO connects them

This is the clever part.

We start with the hypothetical reward:

$$ r(x,y) $$

Bradley-Terry says:

$$ P(y_w\succ y_l) = \sigma(r_w-r_l) $$

Then DPO mathematically shows that under the KL-regularized RL setup:

$$ r(x,y) $$

can be represented using:

$$ \boxed{ \log\frac{\pi(y|x)} {\pi_{\rm ref}(y|x)} } $$

So instead of needing:

Human preference
       ↓
Reward Model
       ↓
reward score

we can use the LLM's own probabilities relative to the reference model.

That's why DPO can directly train the LLM.

The entire thing in baby mode
HUMAN
A > B
 │
 │ "A is better"
 ↓
Bradley-Terry
 │
 │ hypothetical mathematical reward
 ↓
KL-RL math
 │
 │ replace reward with
 │ │
 │ └── LLM probability / reference probability
 ↓
DPO
 │
 ↓
Make chosen response relatively more likely
than rejected response

So your two questions:

"Is the reward just a mathematical thing?"

Yes, in our DPO derivation. It's a latent/hypothetical reward, not a reward model we're training.

"Where did \(\pi\) come from?"

\(\pi\) is simply the LLM policy. Its probabilities come from the model's softmax over tokens.